# Fingerprint Indoor Localization — kNN · RF · MLP

Entrena los tres modelos sobre **todos los experimentos** definidos en `experiments/configs/`
y genera gráficas comparativas.

- **Parte 1** — Entrenamiento y comparación cross-experimento (todos a la vez)
- **Parte 2** — Análisis detallado de un experimento individual

Split estratificado **80/20**, `random_state=123`.

In [ ]:
%matplotlib inline
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "baseline_room.yaml").is_file():
    REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ble_indoor.evaluation.metrics import position_errors_m
from ble_indoor.models.features import position_matrix, rssi_feature_matrix
from ble_indoor.models.fingerprint_knn import FingerprintKnnEstimator
from ble_indoor.models.fingerprint_mlp import FingerprintMlpEstimator
from ble_indoor.models.fingerprint_rf import FingerprintRfEstimator
from ble_indoor.models.knn_zone import ZONE_ID_COLUMN
from ble_indoor.settings import ProjectConfig, ProjectLayout
from ble_indoor.simulation.trace_loader import load_training_trace

sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

MODEL_COLORS = {"kNN": "#4C72B0", "RF": "#55A868", "MLP": "#C44E52"}
layout = ProjectLayout(REPO_ROOT)
CONFIGS_DIR = REPO_ROOT / "experiments" / "configs"
all_config_paths = sorted(CONFIGS_DIR.glob("*.yaml"))
print(f"Experimentos encontrados ({len(all_config_paths)}):")
for p in all_config_paths:
    print(f"  · {p.stem}")

---
## Parte 1 — Entrenamiento de todos los experimentos

Itera sobre cada config, carga su CSV, entrena kNN + RF + MLP y guarda las métricas.

In [ ]:
results = []   # lista de dicts con métricas por (experimento, modelo)
trained = {}   # {exp_name: {"kNN": est, "RF": est, "MLP": est, "val_df": df, "cfg": cfg}}

for cfg_path in all_config_paths:
    exp = cfg_path.stem
    cfg = ProjectConfig.load(cfg_path)
    env = cfg.environment
    csv_path = layout.resolve_repo_path(cfg.training_data.training_trace_csv)

    if not csv_path.is_file():
        print(f"  [SKIP] {exp}: CSV no encontrado ({csv_path})")
        continue

    df = load_training_trace(csv_path, env, cfg.spatial_zones)
    y_zone = df[ZONE_ID_COLUMN].to_numpy(dtype=np.int64)
    counts = np.bincount(y_zone, minlength=cfg.spatial_zones.n_zones)
    strat = y_zone if int(np.min(counts)) >= 5 else None
    train_df, val_df = train_test_split(
        df, test_size=0.20, random_state=123, shuffle=True, stratify=strat
    )

    print(f"\n── {exp}  ({len(train_df)} train / {len(val_df)} val) ──")

    exp_ests = {"val_df": val_df, "cfg": cfg}
    for name, EstClass in [("kNN", FingerprintKnnEstimator),
                           ("RF",  FingerprintRfEstimator),
                           ("MLP", FingerprintMlpEstimator)]:
        est = EstClass.from_config(cfg)
        est.fit(train_df, fit_zone=True)
        m_val = est.evaluate(val_df)
        exp_ests[name] = est
        results.append({
            "experimento": exp,
            "modelo": name,
            "n_train": len(train_df),
            "n_val": len(val_df),
            "zona_acc": m_val["zone"]["accuracy"],
            "rmse_m": m_val["position"]["rmse_xy_m"],
            "mean_m": m_val["position"]["mean_m"],
            "p90_m": m_val["position"]["p90_m"],
        })
        print(f"  {name:4s}  zona={m_val['zone']['accuracy']:.1%}  "
              f"rmse={m_val['position']['rmse_xy_m']:.3f}m")

    trained[exp] = exp_ests

res_df = pd.DataFrame(results)
print("\nEntrenamiento completo.")

## 1.1 Tabla resumen

In [ ]:
pivot = res_df.pivot_table(
    index="experimento",
    columns="modelo",
    values=["zona_acc", "rmse_m", "p90_m"],
    aggfunc="first",
).round(3)
pivot.columns = [f"{v}_{m}" for v, m in pivot.columns]

# ordenar por MLP zona acc descendente
if "zona_acc_MLP" in pivot.columns:
    pivot = pivot.sort_values("zona_acc_MLP", ascending=False)

display_cols = [c for c in pivot.columns if c.startswith("zona_acc")]
try:
    from IPython.display import display as ipy_display
    styled = pivot[display_cols].style \
        .background_gradient(cmap="RdYlGn", axis=None) \
        .format("{:.1%}")
    ipy_display(styled)
    ipy_display(pivot[[c for c in pivot.columns if c.startswith("rmse")]].style.format("{:.3f}"))
except Exception:
    print(pivot.to_string())

## 1.2 Zona accuracy — todos los experimentos y modelos

In [ ]:
exps = res_df["experimento"].unique()
models = ["kNN", "RF", "MLP"]
x = np.arange(len(exps))
bar_w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Zona accuracy
ax = axes[0]
for i, model in enumerate(models):
    sub = res_df[res_df.modelo == model].set_index("experimento")
    vals = [sub.loc[e, "zona_acc"] if e in sub.index else np.nan for e in exps]
    ax.bar(x + (i - 1) * bar_w, vals, bar_w, label=model,
           color=MODEL_COLORS[model], alpha=0.85, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([e.replace("_", "\n") for e in exps], fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("Zona accuracy (val)")
ax.set_title("Zona accuracy por experimento")
ax.legend()

# RMSE
ax = axes[1]
for i, model in enumerate(models):
    sub = res_df[res_df.modelo == model].set_index("experimento")
    vals = [sub.loc[e, "rmse_m"] if e in sub.index else np.nan for e in exps]
    ax.bar(x + (i - 1) * bar_w, vals, bar_w, label=model,
           color=MODEL_COLORS[model], alpha=0.85, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([e.replace("_", "\n") for e in exps], fontsize=8)
ax.set_ylabel("RMSE posición (m)")
ax.set_title("RMSE de posición por experimento")
ax.legend()

plt.suptitle("Comparación todos los experimentos", fontsize=13)
plt.tight_layout()
plt.show()

## 1.3 CDF del error de posición — todos los experimentos

Una subfigura por modelo, una línea por experimento.

In [ ]:
cmap = plt.cm.get_cmap("tab10", len(trained))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, model in zip(axes, ["kNN", "RF", "MLP"]):
    for j, (exp, data) in enumerate(trained.items()):
        est = data[model]
        val_df = data["val_df"]
        cfg = data["cfg"]
        Xv = rssi_feature_matrix(val_df, cfg.environment)
        true_xy = position_matrix(val_df)
        pred_xy = est.predict_xy_batch(Xv)
        err = position_errors_m(true_xy, pred_xy)
        sorted_err = np.sort(err)
        cdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err)
        ax.plot(sorted_err, cdf, label=exp.replace("_", " "),
                color=cmap(j), linewidth=1.6)

    for t in (1.0, 2.0, 3.0):
        ax.axvline(t, color="gray", linestyle=":", linewidth=0.7)
    ax.set_title(f"Modelo: {model}", fontsize=11)
    ax.set_xlabel("Error posición (m)")
    ax.set_xlim(left=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.legend(fontsize=6.5, loc="lower right")

axes[0].set_ylabel("Fracción acumulada")
plt.suptitle("CDF del error de posición — todos los experimentos", fontsize=12)
plt.tight_layout()
plt.show()

## 1.4 Ranking de ganancia: MLP vs kNN y RF

In [ ]:
gain_rows = []
for exp in res_df["experimento"].unique():
    sub = res_df[res_df.experimento == exp].set_index("modelo")
    if all(m in sub.index for m in ["kNN", "RF", "MLP"]):
        gain_rows.append({
            "experimento": exp,
            "MLP vs kNN (zona pp)": (sub.loc["MLP", "zona_acc"] - sub.loc["kNN", "zona_acc"]) * 100,
            "MLP vs RF  (zona pp)": (sub.loc["MLP", "zona_acc"] - sub.loc["RF",  "zona_acc"]) * 100,
            "MLP vs kNN (rmse Δm)": sub.loc["kNN", "rmse_m"] - sub.loc["MLP", "rmse_m"],
            "MLP vs RF  (rmse Δm)": sub.loc["RF",  "rmse_m"] - sub.loc["MLP", "rmse_m"],
        })

gain_df = pd.DataFrame(gain_rows).set_index("experimento")
try:
    from IPython.display import display as ipy_display
    ipy_display(gain_df.style.background_gradient(cmap="RdYlGn", axis=None).format("{:+.2f}"))
except Exception:
    print(gain_df.to_string())

---
## Parte 2 — Análisis detallado de un experimento

Selecciona el experimento a analizar en la siguiente celda.
Usa los modelos ya entrenados en la Parte 1 (no re-entrena).

In [ ]:
# ── elige el experimento para el análisis detallado ──────────────────────────
DETAIL_EXP = "corners_4gw_12x8"
# ─────────────────────────────────────────────────────────────────────────────

assert DETAIL_EXP in trained, f"{DETAIL_EXP} no está en los experimentos entrenados: {list(trained)}"

data    = trained[DETAIL_EXP]
cfg     = data["cfg"]
env     = cfg.environment
val_df  = data["val_df"]

knn = data["kNN"]
rf  = data["RF"]
mlp = data["MLP"]

Xv       = rssi_feature_matrix(val_df, env)
true_xy  = position_matrix(val_df)
z_true   = val_df[ZONE_ID_COLUMN].to_numpy(dtype=np.int64)
zone_labels = cfg.spatial_zones.zone_labels()
label_ids   = list(range(cfg.spatial_zones.n_zones))

pred_xy_knn = knn.predict_xy_batch(Xv)
pred_xy_rf  = rf.predict_xy_batch(Xv)
pred_xy_mlp = mlp.predict_xy_batch(Xv)
z_knn = knn.predict_zone_batch(Xv)
z_rf  = rf.predict_zone_batch(Xv)
z_mlp = mlp.predict_zone_batch(Xv)
err_knn = position_errors_m(true_xy, pred_xy_knn)
err_rf  = position_errors_m(true_xy, pred_xy_rf)
err_mlp = position_errors_m(true_xy, pred_xy_mlp)

print(f"Experimento: {DETAIL_EXP}")
print(f"Val: {len(val_df)} muestras | Zonas: {cfg.spatial_zones.n_zones} | GWs: {env.gateway_ids}")
for name, err in [("kNN", err_knn), ("RF", err_rf), ("MLP", err_mlp)]:
    print(f"  {name:4s}  mediana={np.median(err):.2f}m  p90={np.percentile(err,90):.2f}m")

### CDF + tabla de umbrales

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, err in [("kNN", err_knn), ("RF", err_rf), ("MLP", err_mlp)]:
    se = np.sort(err)
    ax.plot(se, np.arange(1, len(se)+1) / len(se), label=name,
            linewidth=2, color=MODEL_COLORS[name])
for t in (1.0, 2.0, 3.0):
    ax.axvline(t, color="gray", linestyle=":", linewidth=0.8)
ax.set_xlabel("Error de posición (m)")
ax.set_ylabel("Fracción acumulada")
ax.set_title(f"CDF — {DETAIL_EXP}")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend()
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

thresh_rows = []
for name, err in [("kNN", err_knn), ("RF", err_rf), ("MLP", err_mlp)]:
    thresh_rows.append({"modelo": name,
                        **{f"≤{t}m": f"{np.mean(err<=t):.1%}" for t in [1.0,1.5,2.0,3.0]}})
try:
    from IPython.display import display as ipy_display
    ipy_display(pd.DataFrame(thresh_rows).set_index("modelo"))
except Exception:
    print(pd.DataFrame(thresh_rows).to_string(index=False))

### Mapa de predicciones y error espacial

In [ ]:
def draw_room_positions(ax, true_xy, pred_xy, title, color):
    ax.set_aspect("equal", adjustable="box")
    ax.add_patch(plt.Rectangle((0,0), env.room.width_m, env.room.height_m,
                               fill=False, edgecolor="black", linewidth=1.5))
    gw = env.gateway_positions_m()
    ax.scatter(gw[:,0], gw[:,1], c="tab:blue", s=120, zorder=4, marker="^", label="GW")
    ax.scatter(true_xy[:,0], true_xy[:,1], c="#999", s=12, alpha=0.5, label="verdad")
    ax.scatter(pred_xy[:,0], pred_xy[:,1], c=color, s=14, marker="x", alpha=0.7, label="pred")
    for i in range(len(true_xy)):
        ax.plot([true_xy[i,0], pred_xy[i,0]], [true_xy[i,1], pred_xy[i,1]],
                c=color, alpha=0.15, linewidth=0.5)
    ax.set_xlim(-0.5, env.room.width_m+0.5)
    ax.set_ylim(-0.5, env.room.height_m+0.5)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=7)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
draw_room_positions(axes[0], true_xy, pred_xy_knn, "kNN",  MODEL_COLORS["kNN"])
draw_room_positions(axes[1], true_xy, pred_xy_rf,  "RF",   MODEL_COLORS["RF"])
draw_room_positions(axes[2], true_xy, pred_xy_mlp, "MLP",  MODEL_COLORS["MLP"])
plt.suptitle(f"Predicción de posición — {DETAIL_EXP}", fontsize=12)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, err) in zip(axes, [("kNN",err_knn),("RF",err_rf),("MLP",err_mlp)]):
    im = ax.hexbin(true_xy[:,0], true_xy[:,1], C=err, gridsize=20, cmap="magma",
                   reduce_C_function=np.mean,
                   extent=(0,env.room.width_m,0,env.room.height_m))
    ax.set_aspect("equal")
    ax.set_title(f"{name} — error medio (m)")
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    plt.colorbar(im, ax=ax, label="error (m)")
plt.suptitle("Error espacial medio", fontsize=12)
plt.tight_layout()
plt.show()

### Matrices de confusión — zona

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, z_pred, cmap) in zip(axes,
        [("kNN",z_knn,"Blues"),("RF",z_rf,"Greens"),("MLP",z_mlp,"Reds")]):
    cm = confusion_matrix(z_true, z_pred, labels=label_ids)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, linewidths=0.5,
                xticklabels=zone_labels, yticklabels=zone_labels, ax=ax)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Verdadero")
    ax.set_title(f"{name}  (acc={np.trace(cm)/cm.sum():.1%})")
plt.suptitle(f"Confusión zona — {DETAIL_EXP}", fontsize=12)
plt.tight_layout()
plt.show()

### Feature importance — Random Forest

In [ ]:
rssi_cols = [f"rssi_{g}" for g in env.gateway_ids]
if rf._zone is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].barh(rssi_cols, rf._zone.feature_importances_, color="seagreen")
    axes[0].set_title("RF zona — feature_importances_")
    axes[0].set_xlabel("importancia")

    imp_pos = np.mean([t.feature_importances_ for t in rf._position.estimators_], axis=0)
    axes[1].barh(rssi_cols, imp_pos, color="steelblue")
    axes[1].set_title("RF posición — feature_importances_ (media árboles)")
    axes[1].set_xlabel("importancia")
    plt.tight_layout()
    plt.show()